## Manual Prompting Script

This is a "manual prompting" script to serve as a baseline for the byLLM project. Zero-shot prompting is used here, with a subset of the 250-row dataset (row number might be increased in Phase 3) given for the LLM to see how the data is formatted, and it is closer to manual prompting and gives a way to do comparisons with byLLM.

Using Google Gemini's web/chat interface (Gemini 3, Thinking) at gemini.google.com, and these guidelines on prompting a LLM:
- https://github.blog/ai-and-ml/generative-ai/prompt-engineering-guide-generative-ai-llms/
- https://community.ibm.com/community/user/blogs/rashmik-kabiraj/2025/06/03/how-to-make-llm-actually-listen-prompt-engineering
- https://realpython.com/practical-prompt-engineering/
- https://arxiv.org/html/2503.02400v1

#### 2026-02-25 | PROMPT #1:

You are an expert software enginner and data scientist. Your task is to help me build a robust data cleaning and processing pipeline for a dataset that may contain inconsistencies, missing values, or formatting errors. 

I will provide a small sample of the dataset (250 rows) for demonstration purposes. Your pipeline should generalize to larger datasets without exposing the full dataset. 

Requirements:
1. Data Ingestion
   - Suggest and write code to load a dataset (.csv, .json, or similar).
   - Handle potential read errors gracefully.

2. Data Cleaning
   - Detect and repair dirty records, including but not limited to:
       - Inconsistent or invalid date formats
       - Missing or null values in any column
       - Misformatted numeric fields (extra characters, wrong decimal separators)
       - Misformatted categorical fields (typos, inconsistent capitalization)
       - Duplicate rows
       - Trailing or leading whitespace
       - Unexpected or invalid values (e.g., negative ages, impossible categories)
       - Outliers that are likely data entry errors
       - Any other records that might be considered dirty
   - Prioritize high precision (avoid incorrect corrections) and high recall (detect most errors). 
   - Provide clear, step-by-step logic and code for each type of cleaning.

3. Pipeline Skeleton
   - Structure the pipeline with modular functions (e.g., load_data(), clean_data(), analyze_data()). 
   - Include docstrings and comments for maintainability.
   - Include a short demonstration of running the pipeline on the sample dataset. 

4. Validation & Testing
   - Suggest methods to check that cleaning worked correctly (e.g., counts of missing values, sample checks, consistency tests).
   - Optionally, provide small example inputs and expected outputs.

5. Maintainability Notes
   - Explain why this approach is easy to maintain and extend.
   - Highlight potential bottlenecks or limitations.

Output:  
- Python code implementing the pipeline, modular and well-documented.  
- Brief explanation of each module and function.  
- Demonstration of the pipeline applied to the sample dataset.

#### 2026-03-19 | OUTPUT #1:

This response provides a comprehensive Python-based data cleaning and processing pipeline tailored for datasets like the Iowa Liquor Sales data. The pipeline is designed to be modular, robust, and scalable.

**1. Data Cleaning Pipeline Architecture**
The pipeline follows a functional programming paradigm, where the data flows through a series of discrete cleaning steps. This makes it easy to debug, test, and extend.

**2. Implementation Code**

In [1]:
# USER ADDED - time tracking - START - not part of the LLM output here
import time
start_time = time.time() 

In [2]:
import pandas as pd
import numpy as np
import logging
import os

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

class DataPipeline:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    def load_data(self):
        """
        Ingests data from a CSV file.
        Handles missing files and basic parsing errors.
        """
        try:
            logger.info(f"Loading data from {self.file_path}...")
            # Using low_memory=False for better type inference on larger datasets
            self.df = pd.read_csv(self.file_path, low_memory=False)
            logger.info(f"Successfully loaded {len(self.df)} rows.")
            return self.df
        except FileNotFoundError:
            logger.error(f"Error: The file {self.file_path} was not found.")
        except pd.errors.EmptyDataError:
            logger.error("Error: The file is empty.")
        except Exception as e:
            logger.error(f"An unexpected error occurred during ingestion: {e}")
        return None

    def clean_data(self):
        """
        Orchestrates the cleaning process by calling modular sub-functions.
        """
        if self.df is None:
            return None

        self._remove_duplicates()
        self._strip_whitespace()
        self._fix_dates('Date')
        self._handle_missing_values()
        self._normalize_categories(['City', 'County', 'Category Name', 'Item Description'])
        self._clean_numeric_fields(['State Bottle Cost', 'State Bottle Retail', 'Sale (Dollars)'])
        self._validate_business_logic()

        return self.df

    def _remove_duplicates(self):
        initial_count = len(self.df)
        self.df.drop_duplicates(inplace=True)
        dropped = initial_count - len(self.df)
        if dropped > 0:
            logger.info(f"Removed {dropped} duplicate rows.")

    def _strip_whitespace(self):
        """Removes leading/trailing whitespace from all string columns."""
        str_cols = self.df.select_dtypes(include=['object']).columns
        for col in str_cols:
            self.df[col] = self.df[col].astype(str).str.strip()

    def _fix_dates(self, column):
        """Converts column to datetime objects, handling errors gracefully."""
        if column in self.df.columns:
            self.df[column] = pd.to_datetime(self.df[column], errors='coerce')
            null_dates = self.df[column].isna().sum()
            if null_dates > 0:
                logger.warning(f"Detected {null_dates} invalid date formats in '{column}'.")

    def _handle_missing_values(self):
        """
        Drops columns with 100% missing values.
        Fills missing categorical values with 'Unknown'.
        """
        # 1. Drop columns that are entirely null (e.g., County Number in the sample)
        initial_cols = self.df.columns.tolist()
        self.df.dropna(axis=1, how='all', inplace=True)
        dropped_cols = set(initial_cols) - set(self.df.columns)
        if dropped_cols:
            logger.info(f"Dropped empty columns: {dropped_cols}")

        # 2. Fill missing categorical data
        cat_cols = self.df.select_dtypes(include=['object']).columns
        self.df[cat_cols] = self.df[cat_cols].fillna('Unknown')

    def _normalize_categories(self, columns):
        """Ensures consistent casing for categorical strings."""
        for col in columns:
            if col in self.df.columns:
                self.df[col] = self.df[col].str.upper()

    def _clean_numeric_fields(self, columns):
        """Removes symbols ($, ,) and ensures numeric type."""
        for col in columns:
            if col in self.df.columns:
                # Remove currency symbols and commas, then convert
                self.df[col] = self.df[col].replace(r'[\$,]', '', regex=True)
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')

    def _validate_business_logic(self):
        """Detects impossible values (e.g., negative sales or volumes)."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            neg_count = (self.df[col] < 0).sum()
            if neg_count > 0:
                logger.warning(f"Found {neg_count} negative values in {col}. Setting to 0.")
                self.df.loc[self.df[col] < 0, col] = 0

    def analyze_data(self):
        """Basic data profiling after cleaning."""
        if self.df is not None:
            print("\n--- Pipeline Validation Summary ---")
            print(f"Total Rows: {len(self.df)}")
            print(f"Missing Values:\n{self.df.isnull().sum().sum()}")
            print(f"Date Range: {self.df['Date'].min()} to {self.df['Date'].max()}")
            print("-----------------------------------\n")

In [3]:
# --- Execution ---
if __name__ == "__main__":
    pipeline = DataPipeline('../../data/iowa_liquor/iowa_liquor_sales_full.csv')
    raw_data = pipeline.load_data()

    if raw_data is not None:
        cleaned_data = pipeline.clean_data()
        pipeline.analyze_data()

        # Save the result
        cleaned_data.to_csv('../../data/iowa_liquor/mp_cleaned_iowa_liquor_full.csv', index=False)
        print("Cleaned data saved to 'mp_cleaned_iowa_liquor_full.csv'")

INFO: Loading data from ../../data/iowa_liquor/iowa_liquor_sales_full.csv...
INFO: Successfully loaded 27320317 rows.
/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_34102/3203412914.py:60: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = self.df.select_dtypes(include=['object']).columns
/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_34102/3203412914.py:85: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version


--- Pipeline Validation Summary ---
Total Rows: 27320317
Missing Values:
5326663
Date Range: 2012-01-03 00:00:00 to 2025-07-31 00:00:00
-----------------------------------

Cleaned data saved to 'mp_cleaned_iowa_liquor_full.csv'


In [4]:
# USER ADDED - time tracking - STOP - not part of the LLM output here
end_time = time.time()
duration = end_time - start_time
print(f"Execution time: {duration:.2f} seconds")

Execution time: 483.30 seconds


**DRBLITX NOTE:**  
*Important to note that the `pipeline = DataPipeline('iowa_liquor_250.csv')` line was adjusted to add the actual path, which is `../../data/iowa_liquor/iowa_liquor_250.csv`. Same with `clean_data.to_csv('cleaned_liquor_data.csv.csv', index=False)` to ensure the new data was in the correct folder, which is `../../data/iowa_liquor/mp_cleaned_iowa_liquor_250.csv`. Working directory is in /cleaning/fdic_cleaning, hence the ../../.*

**3. Key Pipeline Modules**

* **`load_data`**: Uses `pandas` to read the source file. It includes `try-except` blocks to handle common file system errors, preventing the entire pipeline from crashing if a file is missing or corrupted.
* **`_handle_missing_values`**: Automatically identifies columns that contain no data (like `County Number` in the sample) and removes them to reduce memory usage and noise.
* **`_clean_numeric_fields`**: Uses regex to strip non-numeric characters (like `$` or `,`). This is crucial because data entry often includes formatting that prevents mathematical analysis.
* **`_validate_business_logic`**: This is a "sanity check" layer. It ensures that variables like `Sale (Dollars)` or `Bottles Sold` are not negative, which would represent data entry errors.


**4. Validation & Testing**

To ensure the pipeline worked correctly, we perform three primary checks:

* **Completeness**: `df.isnull().sum()` should show $0$ for critical columns.
* **Consistency**: Verification that `Date` columns are now of type `datetime64[ns]` rather than generic `object` strings.
* **Uniqueness**: `df.duplicated().sum()` should return $0$ to ensure no redundant records remain.


**5. Maintainability and Scalability Notes**

* **Modularity**: Each cleaning step is encapsulated in its own private method (e.g., `_fix_dates`, `_strip_whitespace`). If you need to add a new rule—such as "Zip codes must be $5$ digits"—you simply add one method without altering the core logic.
* **Memory Efficiency**: By dropping empty columns and using `low_memory=False`, the pipeline can be scaled to millions of rows. For extremely large datasets (GBs), you can modify `load_data` to process the file in chunks using `pd.read_csv(chunksize=...)`.
* **Logging**: Instead of simple `print` statements, the use of the `logging` library allows you to redirect error messages to a file for auditing in production environments.
* **Precision vs. Recall**: By using `errors='coerce'` in date and numeric conversions, we prioritize high precision (we do not guess the value if it is garbled). This allows us to easily isolate "dirty" rows for manual review after the automated process.